# Fifth Round of Full Day RFI Flagging Using FRF-Filtered pI SNRs


**by Josh Dillon**, last updated March 10, 2026

This notebook brings together the results of the [single-baseline pI FRF SNR notebook](https://github.com/HERA-Team/hera_notebook_templates/blob/master/notebooks/single_baseline_pI_FRF_SNR.ipynb) to make a set of flagging decisions. This approach is analogous to [Round 4 flagging](https://github.com/HERA-Team/hera_notebook_templates/blob/master/notebooks/full_day_rfi_round_4.ipynb), but uses delay+fringe-rate-filtered pseudo-Stokes pI SNRs instead of delay-only-filtered SNRs. It flags on both coherently rephased SNRs (phased to known bright sources) and incoherently combined SNRs (direction-independent), using Round 4 flags for watershed seeding. 

Here's a set of links to skip to particular figures and tables:
# [• Figure 1: Waterfalls of pI z-Scores Before Round 5 Flagging](#Figure-1:-Waterfalls-of-pI-z-Scores-Before-Round-5-Flagging)
# [• Figure 2: Histogram of z-Scores](#Figure-2:-Histogram-of-z-Scores)
# [• Figure 3: Waterfalls of pI z-Scores After Round 5 Flagging](#Figure-3:-Waterfalls-of-pI-z-Scores-After-Round-5-Flagging)
# [• Figure 4: Summary of Flags Before and After Round 5 Flagging](#Figure-4:-Summary-of-Flags-Before-and-After-Round-5-Flagging)


In [ ]:
import time
tstart = time.time()
!hostname

In [ ]:
import os
os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'
import h5py
import hdf5plugin  # REQUIRED to have the compression plugins available
import numpy as np
import yaml
import glob
import re
import matplotlib
from scipy.signal import convolve, convolve2d
from pyuvdata import UVFlag
from hera_qm import xrfi
from hera_cal import io, flag_utils
from hera_filters import dspec
import matplotlib.pyplot as plt
from astropy.coordinates import SkyCoord, EarthLocation, AltAz
from astropy.time import Time
import astropy.units as u
from astropy import constants as const

from IPython.display import display, HTML
%matplotlib inline
display(HTML("<style>.container { width:100% !important; }</style>"))
_ = np.seterr(all='ignore')  # get rid of red warnings
%config InlineBackend.figure_format = 'retina'

In [ ]:
RED_AVG_FILE = os.environ.get("RED_AVG_FILE", None)
# RED_AVG_FILE = '/lustre/aoc/projects/hera/h6c-analysis/IDR3/2459935/zen.2459935.25792.sum.smooth_calibrated.red_avg.uvh5'

CORNER_TURN_MAP_YAML = os.environ.get("CORNER_TURN_MAP_YAML", 
                                        os.path.join(os.path.dirname(RED_AVG_FILE), "single_baseline_files/corner_turn_map.yaml"))
SNR_SUFFIX = os.environ.get("SNR_SUFFIX", ".inpainted.pI_FRF_SNR.uvh5")
OUTFILE_SUFFIX = os.environ.get("OUTFILE_SUFFIX", ".flag_waterfall_round_5.h5")
jdstr = [s for s in os.path.basename(RED_AVG_FILE).split('.') if s.isnumeric()][0]
OUTFILE_TEMPLATE = os.path.basename(RED_AVG_FILE).split(jdstr)[0] + jdstr + '.{SOURCE_STR}' + OUTFILE_SUFFIX
OUTFILE_TEMPLATE = os.path.join(os.path.dirname(CORNER_TURN_MAP_YAML), OUTFILE_TEMPLATE)

INCOHERENT_SOURCE_STR = "incoherent"

ROUND_4_FLAG_SUFFIX = os.environ.get("ROUND_4_FLAG_SUFFIX", ".flag_waterfall_round_4.h5")

MIN_SAMP_FRAC = float(os.environ.get("MIN_SAMP_FRAC", 0.1))
FM_LOW_FREQ = float(os.environ.get("FM_LOW_FREQ", 87.5)) # in MHz
FM_HIGH_FREQ = float(os.environ.get("FM_HIGH_FREQ", 108.0)) # in MHz

Z_THRESH = float(os.environ.get("Z_THRESH", 4))
WS_Z_THRESH = float(os.environ.get("WS_Z_THRESH", 2))
AVG_Z_THRESH = float(os.environ.get("AVG_Z_THRESH", 1))
MAX_FREQ_FLAG_FRAC = float(os.environ.get("MAX_FREQ_FLAG_FRAC", .25))
MAX_TIME_FLAG_FRAC = float(os.environ.get("MAX_TIME_FLAG_FRAC", .25))
TIME_CONV_SIZE  = float(os.environ.get("TIME_CONV_SIZE", 360)) # in seconds

for setting in ['RED_AVG_FILE', 'CORNER_TURN_MAP_YAML', 'SNR_SUFFIX', 'OUTFILE_SUFFIX', 'OUTFILE_TEMPLATE', 'ROUND_4_FLAG_SUFFIX']:
    print(f'{setting} = "{eval(setting)}"')
for setting in ['MIN_SAMP_FRAC', 'FM_LOW_FREQ', 'FM_HIGH_FREQ', 'Z_THRESH', 'WS_Z_THRESH',
                'AVG_Z_THRESH', 'MAX_FREQ_FLAG_FRAC', 'MAX_TIME_FLAG_FRAC', 'TIME_CONV_SIZE']:
    print(f'{setting} = {eval(setting)}')


In [ ]:
SOURCES = {           # RA,              Dec,              # Search width in hours  
    "PSR J0628-28":   ("06h30m49.3s",    "-28d34m42.6s",   1.2),
    "PSR J0742-2822": ("07h42m49.058s",  "-28d22m43.76s",  1.2),
    "Fornax A":       ("03h22m41.7s",    "-37d12m30s",     2.0),
    "Pictor A":       ("05h19m49.721s",  "-45d46m43.85s",  2.0),
    "Sag A":          ("17h45m40.04s",   "-29d00m28.1s",   2.0),
    "Cen A":          ("13h25m27.6150s", "-43d01m08.806s", 2.0),
    "Cyg A":          ("19h59m28.3566s", "40d44m02.096s",  0.5),
    "Cass A":         ("23h23m26.0s",    "58d48m41s",      0.5),
}

## Load data

In [ ]:
with open(CORNER_TURN_MAP_YAML, 'r') as file:
    corner_turn_map = yaml.unsafe_load(file)

In [ ]:
all_snr_files = [snr_file.replace('.uvh5', SNR_SUFFIX)
                 for snr_files in corner_turn_map['files_to_outfiles_map'].values() 
                 for snr_file in snr_files]
extant_snr_files = [snr_file for snr_file in all_snr_files if os.path.exists(snr_file)]
print(f'Found {len(extant_snr_files)} SNR files, starting with {extant_snr_files[0]}')

In [ ]:
# get autocorrelations
# TODO: generalize for not-inpainted data
all_outfiles = [outfile.replace('.uvh5', '.inpainted.uvh5') for outfiles in corner_turn_map['files_to_outfiles_map'].values() for outfile in outfiles]
for outfile in all_outfiles:
    match = re.search(r'\.(\d+)_(\d+)\.', os.path.basename(outfile))
    if match and match.group(1) == match.group(2):
        hd_autos = io.HERAData(outfile)
        _, _, auto_nsamples = hd_autos.read(polarizations=['ee', 'nn'])
        dt = np.median(np.diff(hd_autos.times))
        break

# For pI data, use the mean of ee and nn auto nsamples as the reference
med_auto_nsamples_pI = np.mean([np.median(auto_nsamples[bl]) for bl in auto_nsamples])

In [ ]:
# load up SNRs, counts, and nsamples
SNRs = {}
SNR_counts = {}
SNR_med_nsamples = {}

# also accumulate incoherent |SNR| sum in the same loop
abs_SNR_sum = None  # initialized on first valid baseline
abs_SNR_count = None

for snr_file in extant_snr_files:
    hd = io.HERADataFastReader(snr_file)
    _, _, nsamples = hd.read(read_data=False, read_flags=False)
    med_nsamples_here = np.max([np.median(n) for n in nsamples.values()])
    if not (med_nsamples_here > MIN_SAMP_FRAC * med_auto_nsamples_pI):
        continue
    
    data, flags, _ = hd.read(read_nsamples=False)
    for bl in data:
        SNRs[bl] = np.where(flags[bl], 0, data[bl])
        SNR_counts[bl] = np.where(flags[bl], 0, 1)
        SNR_med_nsamples[bl] = np.median(nsamples[bl][~flags[bl]])
        
        # accumulate incoherent sum
        if abs_SNR_sum is None:
            abs_SNR_sum = np.zeros_like(SNRs[bl], dtype=float)
            abs_SNR_count = np.zeros_like(SNR_counts[bl], dtype=float)
        abs_SNR_sum += np.abs(SNRs[bl])  # |complex SNR|, Rayleigh-distributed
        abs_SNR_count += SNR_counts[bl]

print(f"Loaded {len(SNRs)} baselines for rephasing and incoherent combination.")


In [ ]:
flag_dir = os.path.dirname(CORNER_TURN_MAP_YAML)
flag_pattern = os.path.join(flag_dir, f'zen.{jdstr}.*{ROUND_4_FLAG_SUFFIX}')
round_4_flag_files = sorted(glob.glob(flag_pattern))

if len(round_4_flag_files) == 0:
    raise ValueError(f'No Round 4 flag files found matching {flag_pattern}')

print(f'Found {len(round_4_flag_files)} Round 4 flag files:')
for f in round_4_flag_files:
    print(f'  {os.path.basename(f)}')

round4_flags = np.any([np.all(UVFlag(flag_file).flag_array, axis=-1)
                        for flag_file in round_4_flag_files], axis=0)
print(f'Combined Round 4 flags: {np.mean(round4_flags):.3%} flagged.')


In [ ]:
# Compute incoherent z-score from |SNR| sum
sigma = 1.0 / np.sqrt(2)  # Rayleigh parameter for |complex noise with unit variance|
rayleigh_mean = sigma * np.sqrt(np.pi / 2)  # = sqrt(pi/4) ~ 0.886
safe_count = np.where(abs_SNR_count > 0, abs_SNR_count, 1)
variance_expected = (4 - np.pi) / 2 * sigma**2 / safe_count
zscore_incoherent = (abs_SNR_sum / safe_count - rayleigh_mean) / variance_expected**.5
zscore_incoherent = np.where(abs_SNR_count == 0, np.nan, zscore_incoherent)

# Recenter per band for robustness
_, (low_band, high_band) = flag_utils.get_minimal_slices(
    ~np.isfinite(zscore_incoherent), freqs=data.freqs,
    freq_cuts=[(FM_LOW_FREQ + FM_HIGH_FREQ) * .5e6])
for band in [low_band, high_band]:
    zscore_incoherent[:, band] -= np.nanmedian(zscore_incoherent[:, band])

# Apply Round 4 flags for watershed seeding
assert round4_flags.shape == (len(hd.times), len(hd.freqs)), \
    f"Round 4 flag shape {round4_flags.shape} doesn't match data shape {(len(hd.times), len(hd.freqs))}"
zscore_incoherent[round4_flags] = np.nan
print(f'Incoherent z-score computed: {np.nanmedian(zscore_incoherent):.3f} median, {np.nanstd(zscore_incoherent):.3f} std.')


## Plotting Functions

In [ ]:
def plot_z_score(zscore, source, source_name, flags=None, vmin=-Z_THRESH, vmax=Z_THRESH, in_range=None, label=None):
    if in_range is None:
        in_range=slice(None)
    
    if flags is None:
        flags = ~np.isfinite(zscore)
    plt.figure(figsize=(14,6), dpi=300)
    extent = [data.freqs[0] / 1e6, data.freqs[-1] / 1e6, 
              data.times[in_range][-1] - int(data.times[0]), data.times[in_range][0] - int(data.times[0])]

    if label is None:
        label=f'pI FRF-Filtered z-score Phased to {source_name} and Coherently Averaged Across Baselines'
    
    plt.imshow(np.where(flags, np.nan, zscore)[in_range, :], aspect='auto', 
               cmap='coolwarm', interpolation='none', vmin=vmin, vmax=vmax, extent=extent)
    plt.colorbar(location='top', extend='both', aspect=40, pad=.02, label=label)
    plt.xlabel('Frequency (MHz)')
    plt.ylabel(f'JD - {int(data.times[0])}')
    plt.tight_layout()
    
    # Add LST right axis with proper wrapping
    lst_grid = hd.lsts[in_range] * 12 / np.pi  # radians to hours
    lst_grid[lst_grid > lst_grid[-1]] -= 24
    ax2 = plt.gca().twinx()
    ax2.set_ylim(lst_grid[-1], lst_grid[0])
    mod24 = lambda x, _: f"{x % 24:.1f}"
    ax2.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(mod24))
    ax2.set_ylabel('LST (hours)')        
    
    # Mark transit LST on the right axis (only for rephased sources)
    if source is not None:
        transit_lst = source.ra.rad * 12 / np.pi
        # Apply the same 24h wrapping used for the LST grid
        if transit_lst > (hd.lsts[in_range][-1] * 12 / np.pi):
            transit_lst -= 24

        lst_lo, lst_hi = sorted(ax2.get_ylim())
        x_pos = data.freqs[-1] / 1e6  # right edge, next to LST axis

        if lst_lo <= transit_lst <= lst_hi:
            ax2.plot(x_pos, transit_lst, 'ko', markersize=10, clip_on=False,
                     label=f'{source_name} transit')
        elif transit_lst > lst_hi:
            ax2.plot(x_pos, lst_hi, marker='^', color='k', markersize=10,
                     clip_on=False, label=f'{source_name} transit (above)')
        else:
            ax2.plot(x_pos, lst_lo, marker='v', color='k', markersize=10,
                     clip_on=False, label=f'{source_name} transit (below)')
        ax2.legend(loc='upper right')
    
    plt.show()


In [ ]:
def plot_histogram(zscore_per_source, in_range_per_source={}, zscore_incoherent=None):
    plt.figure(figsize=(14,4), dpi=100)
    bins = np.arange(-3, 15, .1)
    
    for source_name, zscore in zscore_per_source.items():
        in_range = in_range_per_source.get(source_name, slice(None))
        hist = plt.hist(np.ravel(zscore[in_range, :]), bins=bins, density=True, label=f'Phased to {source_name}', alpha=.75, histtype='step', zorder=100)
    
    if zscore_incoherent is not None:
        hist = plt.hist(np.ravel(zscore_incoherent), bins=bins, density=True,
                        label='Incoherent Average', alpha=.9, histtype='step',
                        color='k', lw=2, zorder=150)
    
    # Standardized Rayleigh
    mu_r = np.sqrt(np.pi / 2)
    sig_r = np.sqrt((4 - np.pi) / 2)
    x_r = sig_r * bins + mu_r
    pdf_r = np.where(x_r >= 0,
        sig_r * x_r * np.exp(-0.5 * x_r**2),
        0.0)
    plt.plot(bins, pdf_r, 'k--', label='Shifted and Scaled Rayleigh', zorder=0)
    
    plt.axvline(WS_Z_THRESH, c='r', ls='--', label='Watershed z-score')
    plt.axvline(Z_THRESH, c='r', ls='-', label='Threshold z-score')    
    plt.yscale('log')
    all_densities = hist[0][hist[0] > 0]
    plt.ylim(np.min(all_densities) / 2, np.max(all_densities) * 2)
    plt.xlim([-3, 15])
    plt.legend()
    plt.xlabel('z-score')
    plt.ylabel('Density')
    plt.tight_layout()


In [ ]:
def summarize_flagging(zscore_per_source, flags_per_source, flags_incoherent=None):

    prior_flags = np.all(~np.isfinite(list(zscore_per_source.values())), axis=0)
    if flags_incoherent is not None:
        prior_flags &= ~np.isfinite(zscore_incoherent)
    
    to_plot = np.zeros(prior_flags.shape, dtype=float)  # no flags
    flag_sum = np.sum(list(flags_per_source.values()), axis=0)
    if flags_incoherent is not None:
        flag_sum_with_incoh = flag_sum + flags_incoherent.astype(int)
    else:
        flag_sum_with_incoh = flag_sum

    n_sources = len(flags_per_source)
    for i, (source_name, flags) in enumerate(flags_per_source.items()):
        to_plot[((flag_sum_with_incoh) == 1) & flags] = i + 2  # flags attributable to a single source
    if flags_incoherent is not None:
        to_plot[((flag_sum_with_incoh) == 1) & flags_incoherent] = n_sources + 2  # incoherent-only flags
    to_plot[flag_sum_with_incoh >= 2] = n_sources + 3  # flags with multiple sources/methods
    to_plot[prior_flags] = 1  # prior flags
    
    plt.figure(figsize=(14,10), dpi=300)
    n_categories = n_sources + 4  # unflagged, prior, per-source..., incoherent, multiple
    if flags_incoherent is None:
        n_categories = n_sources + 3
    colors = list(((0, 0, 0), (.5, .5, .5)) + matplotlib.colormaps["Set2"].colors[0:n_sources])
    if flags_incoherent is not None:
        colors.append(matplotlib.colormaps['Set1'].colors[0])  # red for incoherent
    colors.append((1, 1, 1))  # white for multiple
    cmap = matplotlib.colors.ListedColormap(colors)
    extent = [data.freqs[0] / 1e6, data.freqs[-1] / 1e6, 
              data.times[-1] - int(data.times[0]), data.times[0] - int(data.times[0])]    
    plt.imshow(to_plot,aspect='auto', cmap=cmap, interpolation='none', extent=extent)
    plt.clim([-.5, n_categories - .5])
    cbar = plt.colorbar(location='top', aspect=40, pad=.02)
    cbar.set_ticks(list(range(n_categories)))
    tick_labels = ['Unflagged', 'Flagged After Round 4'] + [f'Flags from\n{source_name}' for source_name in flags_per_source]
    if flags_incoherent is not None:
        tick_labels.append('Flags from\nIncoherent')
    tick_labels.append('Flags from\nMultiple Sources')
    cbar.set_ticklabels(tick_labels)
    plt.xlabel('Frequency (MHz)')
    plt.ylabel(f'JD - {int(data.times[0])}')

    # Add LST right axis with proper wrapping
    lst_grid = hd.lsts * 12 / np.pi  # radians to hours
    lst_grid[lst_grid > lst_grid[-1]] -= 24
    ax2 = plt.gca().twinx()
    ax2.set_ylim(lst_grid[-1], lst_grid[0])
    mod24 = lambda x, _: f"{x % 24:.1f}"
    ax2.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(mod24))
    ax2.set_ylabel('LST (hours)')        
    plt.tight_layout()


## Rephasing

In [ ]:
zscore_per_source = {}
in_range_per_source = {}
for source_name in SOURCES:

    print(f'Now computing SNRs rephased to {source_name}.')
    
    # Source and observatory
    source = SkyCoord(*SOURCES[source_name][0:2])
    location = EarthLocation(
        lat=hd.info['latitude'] * u.deg,
        lon=hd.info['longitude'] * u.deg,
        height=hd.info['altitude'] * u.m,
    )    

    # Transform to AltAz at each observation time
    times = Time(hd.times, format='jd')
    source_altaz = source.transform_to(AltAz(obstime=times, location=location))    

    # Build ENU unit vectors from alt/az
    az = source_altaz.az.rad
    alt = source_altaz.alt.rad

    # figure out if any times are within range
    diff = (hd.lsts * 12 / np.pi - source.ra.deg * 24 / 360) % 24
    in_range = np.minimum(diff, 24 - diff) < (SOURCES[source_name][2] / 2)
    if np.sum(in_range) == 0:
        print(f'\t{source_name} is never within {SOURCES[source_name][2]} hours of transit on this JD.')
        continue
    else:
        print(f'\t{np.sum(in_range)} integrations are within {SOURCES[source_name][2]} hours of the transit of {source_name}.')
        in_range_per_source[source_name] = in_range
    
    s_enu = np.column_stack([np.sin(az) * np.cos(alt), np.cos(az) * np.cos(alt), np.sin(alt)])
    # s_diff is (zenith - source) because we're *removing* the source's geometric phase,
    # as opposed to hera_cal.utils.lst_rephase which *applies* phase with (new_zenith - old_zenith).
    s_diff_over_c = (np.array([0., 0., 1.]) - s_enu) / const.c.value
    
    SNR_sum = np.zeros((len(hd.times), len(hd.freqs)), dtype=complex)
    SNR_count = np.zeros((len(hd.times), len(hd.freqs)), dtype=float)
    bls_used = []
    for bl in SNRs:
        if np.median(SNR_med_nsamples[bl]) > MIN_SAMP_FRAC * med_auto_nsamples_pI:
            bl_len = np.linalg.norm(hd.antpos[bl[0]] - hd.antpos[bl[1]])
            if (bl_len > 1):
                bls_used.append(bl)
                bl_vec = hd.antpos[bl[0]] - hd.antpos[bl[1]]  # this is the hera_cal convention that matches utils.lst_rephase
                tau = np.einsum('ti,i->t', s_diff_over_c, bl_vec)
                phs = np.exp(-2j * np.pi * hd.freqs[np.newaxis, :] * tau[:, np.newaxis])  # this also matches the hera_cal convention
                SNR_sum += SNRs[bl] * phs
                SNR_count += SNR_counts[bl]


    sigma = 1.0 / np.sqrt(2)  # Rayleigh parameter for |complex noise with unit variance|
    rayleigh_mean = sigma * np.sqrt(np.pi / 2)  # = √(π/4) ≈ 0.886
    safe_count = np.where(SNR_count > 0, SNR_count, 1)
    predicted_mean = rayleigh_mean / np.sqrt(safe_count)
    variance_expected = (4 - np.pi) / 2 * sigma**2 / safe_count
    zscore_per_source[source_name] = (np.abs(SNR_sum) / safe_count - predicted_mean) / variance_expected**.5
    zscore_per_source[source_name] = np.where(SNR_count == 0, np.nan, zscore_per_source[source_name])  

    # Apply Round 4 flags for watershed seeding
    assert round4_flags.shape == (len(hd.times), len(hd.freqs)), \
        f"Round 4 flag shape {round4_flags.shape} doesn't match data shape {(len(hd.times), len(hd.freqs))}"
    zscore_per_source[source_name][round4_flags] = np.nan


## Flagging

In [ ]:
def iteratively_flag_on_averaged_zscore(flags, zscore, avg_func=np.nanmean, avg_z_thresh=AVG_Z_THRESH, verbose=True):
    '''Flag whole integrations or channels based on average z-score. This is done
    iteratively to prevent bad times affecting channel averages or vice versa.'''

    _, (low_band, high_band) = flag_utils.get_minimal_slices(flags, freqs=data.freqs, freq_cuts=[(FM_LOW_FREQ + FM_HIGH_FREQ) * .5e6])
    flagged_chan_count = 0
    flagged_int_count = {low_band: 0, high_band: 0}
    for band in (low_band, high_band):
        while True:
            zspec = avg_func(np.where(flags, np.nan, zscore)[:, band], axis=0)
            ztseries = avg_func(np.where(flags, np.nan, zscore)[:, band], axis=1)
    
            if (np.nanmax(zspec) < avg_z_thresh) and (np.nanmax(ztseries[:]) < avg_z_thresh):
                break
    
            if np.nanmax(zspec) >= np.nanmax(ztseries[:]):
                flagged_chan_count += np.sum((zspec >= np.nanmax(ztseries)) & (zspec >= avg_z_thresh))
                flags[:, band][:, (zspec >= np.nanmax(ztseries)) & (zspec >= avg_z_thresh)] = True
            else:
                flagged_int_count[band] += np.sum((ztseries >= np.nanmax(zspec)) & (ztseries >= avg_z_thresh))
                flags[(ztseries >= np.nanmax(zspec)) & (ztseries >= avg_z_thresh), band] = True

    ztseries_low = avg_func(np.where(flags, np.nan, zscore)[:, low_band], axis=1)
    flags[(ztseries_low > avg_z_thresh) & np.all(flags[:, high_band], axis=1), low_band] = True
    
    if verbose:
        if (flagged_int_count[low_band] > 0) or (flagged_int_count[high_band] > 0) or (flagged_chan_count > 0):
            print(f'\t\tFlagging an additional {flagged_int_count[low_band]} low-band integrations, '
                  f'{flagged_int_count[high_band]} high-band integrations, and {flagged_chan_count} channels.')

def impose_max_chan_flag_frac(flags, max_flag_frac=MAX_FREQ_FLAG_FRAC, verbose=True):
    '''Flag channels already flagged more than max_flag_frac (excluding completely flagged times).'''
    _, (low_band, high_band) = flag_utils.get_minimal_slices(flags, freqs=data.freqs, freq_cuts=[(FM_LOW_FREQ + FM_HIGH_FREQ) * .5e6])
    for band in [low_band, high_band]:
        unflagged_times = ~np.all(flags[:, band], axis=1)
        frequently_flagged_chans =  np.mean(flags[unflagged_times, band], axis=0) >= max_flag_frac
        if verbose:
            flag_diff_count = np.sum(frequently_flagged_chans) - np.sum(np.all(flags[:, band], axis=0))
            if flag_diff_count > 0:
                print(f'\tFlagging {flag_diff_count} channels previously flagged {max_flag_frac:.2%} or more.')        
        flags[:, band][:, frequently_flagged_chans] = True
        
def impose_max_time_flag_frac(flags, max_flag_frac=MAX_TIME_FLAG_FRAC, verbose=True):
    '''Flag times already flagged more than max_flag_frac (excluding completely flagged channels).'''
    _, (low_band, high_band) = flag_utils.get_minimal_slices(flags, freqs=data.freqs, freq_cuts=[(FM_LOW_FREQ + FM_HIGH_FREQ) * .5e6])
    for name, band in zip(['low', 'high'], [low_band, high_band]):
        unflagged_chans = ~np.all(flags[:, band], axis=0)
        frequently_flagged_times =  np.mean(flags[:, band][:, unflagged_chans], axis=1) >= max_flag_frac
        if verbose:
            flag_diff_count = np.sum(frequently_flagged_times) - np.sum(np.all(flags[:, band], axis=1))
            if flag_diff_count > 0:
                print(f'\t\tFlagging {flag_diff_count} {name}-band times previously flagged {max_flag_frac:.2%} or more.')
        flags[frequently_flagged_times, band] = True

def watershed_flag(flags, zscore, ws_z_thresh=WS_Z_THRESH):
    '''Wrapper around xrfi._ws_flag_waterfall to be performed separately above and below FM.'''
    while True:        
        nflags = np.sum(flags)
        _, (low_band, high_band) = flag_utils.get_minimal_slices(flags, freqs=data.freqs, freq_cuts=[(FM_LOW_FREQ + FM_HIGH_FREQ) * .5e6])
        for band in [low_band, high_band]:
            flags[:, band] |= (xrfi._ws_flag_waterfall(zscore[:, band], flags[:, band], ws_z_thresh))
        if np.sum(flags) == nflags:
            break

def iterative_time_conv_flagging(flags, zscore, conv_size=TIME_CONV_SIZE, one_time_thresh=Z_THRESH, full_kernel_thresh=AVG_Z_THRESH):
    '''Looks for streteches of increasing size that fit a decreasing threshold. At conv_size (in seconds), it flags 
    stretches with average z-score above full_kernel_thresh. At one pixel, it uses one_time_thresh.
    In between, it interpolates logarithmically.'''
    dt_s = dt * 3600 * 24
    heights = np.array([int(h) + 1 for h in 2**np.arange(1, np.ceil(np.log2(conv_size / dt_s) + np.finfo(float).eps))])
    
    # Create cuts that get more strict as the kernel gets bigger
    cuts = one_time_thresh * (full_kernel_thresh / one_time_thresh)**((heights - 1) / (conv_size / dt_s - 1))

    for height, cut in zip(heights, cuts):
        result = {}
        kernel = np.ones((int(height), 1), dtype=float)
        mask = ~(np.isnan(zscore) | flags)
        filled_data = np.where(mask, zscore, 0.0)
        conv_data = convolve2d(filled_data, kernel, mode='same')
        conv_mask = convolve2d(mask.astype(float), kernel, mode='same')
        with np.errstate(divide='ignore', invalid='ignore'):
            result = conv_data / conv_mask

        above_cut = result > cut 
        flags[:, :] |= (convolve2d(above_cut.astype(float), kernel, mode='same') > 0)
        
        print(f'\t\t{np.mean(flags):.3%} of waterfall flagged after {height}-integration convolution-based flagging with z-scores above {cut:.3f}.')

In [ ]:
flags_per_source = {}

for source_name, zscore in zscore_per_source.items():
    print(f'Now building flagging mask on SNRs rephased to {source_name}...')
    flags = ~np.isfinite(zscore)
    _, (low_band, high_band) = flag_utils.get_minimal_slices(flags, freqs=data.freqs, freq_cuts=[(FM_LOW_FREQ + FM_HIGH_FREQ) * .5e6])    
    print(f'\t{np.mean(flags):.3%} of waterfall flagged to start.')

    # flag whole integrations or channels using outliers in median
    while True:
        nflags = np.sum(flags)  
        iteratively_flag_on_averaged_zscore(flags, zscore, avg_func=np.nanmedian, avg_z_thresh=AVG_Z_THRESH, verbose=True)
        impose_max_chan_flag_frac(flags, max_flag_frac=MAX_FREQ_FLAG_FRAC, verbose=True)
        impose_max_time_flag_frac(flags, max_flag_frac=MAX_TIME_FLAG_FRAC, verbose=True)
        if np.sum(flags) == nflags:
            break  
    print(f'\t{np.mean(flags):.3%} of waterfall flagged after flagging whole times and channels with median z > {AVG_Z_THRESH}.')
    
    for band in [low_band, high_band]:
        flags[:, band] |= (zscore[:, band] > Z_THRESH) 
    print(f'\t{np.mean(flags):.3%} of waterfall flagged after flagging z > {Z_THRESH} outliers.')

    # iterative time-convolved flagging
    iterative_time_conv_flagging(flags, zscore, conv_size=TIME_CONV_SIZE, one_time_thresh=Z_THRESH, full_kernel_thresh=AVG_Z_THRESH)
    print(f'\t{np.mean(flags):.3%} of waterfall flagged after time convolution flagging.')

    # watershed flagging 
    watershed_flag(flags, zscore, ws_z_thresh=WS_Z_THRESH)
    print(f'\t{np.mean(flags):.3%} of waterfall flagged after watershed flagging on z > {WS_Z_THRESH} neighbors of prior flags (seeded with Round 4 flags).')

    # flag whole integrations or channels using outliers in mean
    while True:
        nflags = np.sum(flags)
        iteratively_flag_on_averaged_zscore(flags, zscore, avg_func=np.nanmean, avg_z_thresh=AVG_Z_THRESH, verbose=True)
        impose_max_chan_flag_frac(flags, max_flag_frac=MAX_FREQ_FLAG_FRAC, verbose=True)
        impose_max_time_flag_frac(flags, max_flag_frac=MAX_TIME_FLAG_FRAC, verbose=True)
        if np.sum(flags) == nflags:
            break  
    print(f'\t{np.mean(flags):.3%} of waterfall flagged after flagging whole times and channels with average z > {AVG_Z_THRESH}.')
    
    # watershed flagging one last time
    watershed_flag(flags, zscore, ws_z_thresh=WS_Z_THRESH)
    print(f'\t{np.mean(flags):.3%} of waterfall flagged after watershed flagging again on z > {WS_Z_THRESH} neighbors of prior flags.')    

    flags_per_source[source_name] = np.where(in_range_per_source[source_name][:, None], flags, ~np.isfinite(zscore))
    flags_per_source[source_name] = np.where(np.all(flags, axis=0)[None, :], flags, flags_per_source[source_name])
    print('')

## Incoherent Flagging


In [ ]:
print('Now building flagging mask on incoherently combined SNRs...')
flags_incoherent = ~np.isfinite(zscore_incoherent)
_, (low_band, high_band) = flag_utils.get_minimal_slices(flags_incoherent, freqs=data.freqs, freq_cuts=[(FM_LOW_FREQ + FM_HIGH_FREQ) * .5e6])    
print(f'\t{np.mean(flags_incoherent):.3%} of waterfall flagged to start.')

# flag whole integrations or channels using outliers in median
while True:
    nflags = np.sum(flags_incoherent)  
    iteratively_flag_on_averaged_zscore(flags_incoherent, zscore_incoherent, avg_func=np.nanmedian, avg_z_thresh=AVG_Z_THRESH, verbose=True)
    impose_max_chan_flag_frac(flags_incoherent, max_flag_frac=MAX_FREQ_FLAG_FRAC, verbose=True)
    impose_max_time_flag_frac(flags_incoherent, max_flag_frac=MAX_TIME_FLAG_FRAC, verbose=True)
    if np.sum(flags_incoherent) == nflags:
        break  
print(f'\t{np.mean(flags_incoherent):.3%} of waterfall flagged after flagging whole times and channels with median z > {AVG_Z_THRESH}.')

for band in [low_band, high_band]:
    flags_incoherent[:, band] |= (zscore_incoherent[:, band] > Z_THRESH) 
print(f'\t{np.mean(flags_incoherent):.3%} of waterfall flagged after flagging z > {Z_THRESH} outliers.')

# iterative time-convolved flagging
iterative_time_conv_flagging(flags_incoherent, zscore_incoherent, conv_size=TIME_CONV_SIZE, one_time_thresh=Z_THRESH, full_kernel_thresh=AVG_Z_THRESH)
print(f'\t{np.mean(flags_incoherent):.3%} of waterfall flagged after time convolution flagging.')

# watershed flagging 
watershed_flag(flags_incoherent, zscore_incoherent, ws_z_thresh=WS_Z_THRESH)
print(f'\t{np.mean(flags_incoherent):.3%} of waterfall flagged after watershed flagging on z > {WS_Z_THRESH} neighbors of prior flags (seeded with Round 4 flags).')

# flag whole integrations or channels using outliers in mean
while True:
    nflags = np.sum(flags_incoherent)
    iteratively_flag_on_averaged_zscore(flags_incoherent, zscore_incoherent, avg_func=np.nanmean, avg_z_thresh=AVG_Z_THRESH, verbose=True)
    impose_max_chan_flag_frac(flags_incoherent, max_flag_frac=MAX_FREQ_FLAG_FRAC, verbose=True)
    impose_max_time_flag_frac(flags_incoherent, max_flag_frac=MAX_TIME_FLAG_FRAC, verbose=True)
    if np.sum(flags_incoherent) == nflags:
        break  
print(f'\t{np.mean(flags_incoherent):.3%} of waterfall flagged after flagging whole times and channels with average z > {AVG_Z_THRESH}.')

# watershed flagging one last time
watershed_flag(flags_incoherent, zscore_incoherent, ws_z_thresh=WS_Z_THRESH)
print(f'\t{np.mean(flags_incoherent):.3%} of waterfall flagged after watershed flagging again on z > {WS_Z_THRESH} neighbors of prior flags.')


# Figure 1: Waterfalls of pI z-Scores Before Round 5 Flagging


In [ ]:
for source_name, zscore in zscore_per_source.items():
    source = SkyCoord(*SOURCES[source_name][0:2])
    plot_z_score(zscore_per_source[source_name], source, source_name, in_range=in_range_per_source[source_name])

# Incoherent z-score waterfall
plot_z_score(zscore_incoherent, None, 'Incoherent',
             label='pI FRF-Filtered z-score Incoherently Averaged Across Baselines')


# Figure 2: Histogram of z-Scores


In [ ]:
plot_histogram(zscore_per_source, in_range_per_source=in_range_per_source, zscore_incoherent=zscore_incoherent)


# Figure 3: Waterfalls of pI z-Scores After Round 5 Flagging


In [ ]:
for source_name, zscore in zscore_per_source.items():
    source = SkyCoord(*SOURCES[source_name][0:2])
    plot_z_score(zscore, source, source_name, flags=flags_per_source[source_name], in_range=in_range_per_source[source_name])

# Incoherent z-score waterfall after flagging
plot_z_score(zscore_incoherent, None, 'Incoherent', flags=flags_incoherent,
             label='pI FRF-Filtered z-score Incoherently Averaged Across Baselines')


# Figure 4: Summary of Flags Before and After Round 5 Flagging


In [ ]:
summarize_flagging(zscore_per_source, flags_per_source, flags_incoherent=flags_incoherent)


## Save results

In [ ]:
uvf = UVFlag(hd_autos, mode='flag', waterfall=True)

# Save per-source rephased flags
for source_name, flags in flags_per_source.items():
    for polind in range(uvf.flag_array.shape[2]):
        uvf.flag_array[:, :, polind] = flags

    outfile = OUTFILE_TEMPLATE.format(SOURCE_STR=source_name.replace(" ", "_"))
    print(f'Now writing {outfile}')
    uvf.write(outfile, clobber=True)

# Save incoherent flags
for polind in range(uvf.flag_array.shape[2]):
    uvf.flag_array[:, :, polind] = flags_incoherent
outfile = OUTFILE_TEMPLATE.format(SOURCE_STR=INCOHERENT_SOURCE_STR)
print(f'Now writing {outfile}')
uvf.write(outfile, clobber=True)


## Metadata

In [ ]:
for repo in ['hera_cal', 'hera_qm', 'hera_filters', 'hera_notebook_templates', 'pyuvdata']:
    exec(f'from {repo} import __version__')
    print(f'{repo}: {__version__}')

In [ ]:
print(f'Finished execution in {(time.time() - tstart) / 60:.2f} minutes.')